# 05 — Incremental Load + Idempotency

Purpose: process only new events and make the pipeline safe to run again.

Process schema:

```text
EXISTING SILVER TABLE
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e1       | u1      | 10.0   |
| e2       | u2      | 20.0   |
+----------+---------+--------+

NEW BATCH
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e2       | u2      | 20.0   | already processed
| e3       | u3      | 30.0   | new
| e4       | u4      | 40.0   | new
+----------+---------+--------+

                         ANTI JOIN on event_id
                                  |
                                  v

NEW EVENTS ONLY
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e3       | u3      | 30.0   |
| e4       | u4      | 40.0   |
+----------+---------+--------+

                         UNION into Silver
                                  |
                                  v

UPDATED SILVER TABLE
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e1       | u1      | 10.0   |
| e2       | u2      | 20.0   |
| e3       | u3      | 30.0   |
| e4       | u4      | 40.0   |
+----------+---------+--------+
```

Idempotency means:

```text
Running the same batch twice produces the same final result.
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-incremental-idempotency")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Existing Silver table

This simulates data that was already processed earlier.

In [2]:
silver_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_time_raw", StringType(), False),
])

existing_rows = [
    ("e1", "u1", 10.0, "2026-01-01 10:00:00"),
    ("e2", "u2", 20.0, "2026-01-01 10:05:00"),
]

existing_silver = (
    spark.createDataFrame(existing_rows, silver_schema)
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .drop("event_time_raw")
)

existing_silver.show(truncate=False)

+--------+-------+------+-------------------+
|event_id|user_id|amount|event_time         |
+--------+-------+------+-------------------+
|e1      |u1     |10.0  |2026-01-01 10:00:00|
|e2      |u2     |20.0  |2026-01-01 10:05:00|
+--------+-------+------+-------------------+



## Step 2 — New incoming batch

The new batch contains:

```text
e2 = already processed
e3 = new
e4 = new
```

In [3]:
batch_rows = [
    ("e2", "u2", 20.0, "2026-01-01 10:05:00"),
    ("e3", "u3", 30.0, "2026-01-01 10:10:00"),
    ("e4", "u4", 40.0, "2026-01-01 10:15:00"),
]

incoming_batch = (
    spark.createDataFrame(batch_rows, silver_schema)
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .drop("event_time_raw")
)

incoming_batch.show(truncate=False)

+--------+-------+------+-------------------+
|event_id|user_id|amount|event_time         |
+--------+-------+------+-------------------+
|e2      |u2     |20.0  |2026-01-01 10:05:00|
|e3      |u3     |30.0  |2026-01-01 10:10:00|
|e4      |u4     |40.0  |2026-01-01 10:15:00|
+--------+-------+------+-------------------+



## Step 3 — Detect already processed rows

Use `left_anti` join.

```text
incoming_batch LEFT ANTI JOIN existing_silver ON event_id
```

Result:

```text
Only rows from incoming_batch that do not exist in existing_silver.
```

In [4]:
new_events_only = (
    incoming_batch.alias("new")
    .join(
        existing_silver.select("event_id").alias("existing"),
        on="event_id",
        how="left_anti"
    )
)

new_events_only.show(truncate=False)

+--------+-------+------+-------------------+
|event_id|user_id|amount|event_time         |
+--------+-------+------+-------------------+
|e3      |u3     |30.0  |2026-01-01 10:10:00|
|e4      |u4     |40.0  |2026-01-01 10:15:00|
+--------+-------+------+-------------------+



## Step 4 — Append only new rows

For this simple notebook we use `unionByName`.

In a real table this would usually be a Delta `MERGE` or an append guarded by keys.

In [5]:
updated_silver = existing_silver.unionByName(new_events_only)

updated_silver.orderBy("event_id").show(truncate=False)

+--------+-------+------+-------------------+
|event_id|user_id|amount|event_time         |
+--------+-------+------+-------------------+
|e1      |u1     |10.0  |2026-01-01 10:00:00|
|e2      |u2     |20.0  |2026-01-01 10:05:00|
|e3      |u3     |30.0  |2026-01-01 10:10:00|
|e4      |u4     |40.0  |2026-01-01 10:15:00|
+--------+-------+------+-------------------+



## Step 5 — Rerun the same batch

This simulates running the same pipeline again with the same input batch.

Expected result:

```text
No new rows should be added.
```

In [6]:
rerun_new_events_only = (
    incoming_batch.alias("new")
    .join(
        updated_silver.select("event_id").alias("existing"),
        on="event_id",
        how="left_anti"
    )
)

rerun_new_events_only.show(truncate=False)
print("Rows to add on rerun:", rerun_new_events_only.count())

+--------+-------+------+----------+
|event_id|user_id|amount|event_time|
+--------+-------+------+----------+
+--------+-------+------+----------+

Rows to add on rerun: 0


## Step 6 — Control totals

The rerun should not change the final row count.

In [7]:
after_rerun_silver = updated_silver.unionByName(rerun_new_events_only)

control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("existing_silver_rows", existing_silver.count()),
        ("incoming_batch_rows", incoming_batch.count()),
        ("first_run_new_rows", new_events_only.count()),
        ("updated_silver_rows", updated_silver.count()),
        ("rerun_new_rows", rerun_new_events_only.count()),
        ("after_rerun_silver_rows", after_rerun_silver.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+-----------------------+-----+
|metric                 |value|
+-----------------------+-----+
|existing_silver_rows   |2    |
|incoming_batch_rows    |3    |
|first_run_new_rows     |2    |
|updated_silver_rows    |4    |
|rerun_new_rows         |0    |
|after_rerun_silver_rows|4    |
+-----------------------+-----+



## Step 7 — Duplicate check

Final Silver should have one row per `event_id`.

In [8]:
duplicate_check = (
    after_rerun_silver
    .groupBy("event_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
)

duplicate_check.show(truncate=False)

+--------+---------+
|event_id|row_count|
+--------+---------+
+--------+---------+



## Final result

```text
INCOMING BATCH
      |
      v
LEFT ANTI JOIN against existing Silver keys
      |
      +--> NEW EVENTS ONLY
      |
      v
APPEND / MERGE INTO SILVER
      |
      v
SAFE RERUN
```

The key idea:

```text
Do not blindly append a batch.
First compare it with already processed keys.
```

In [9]:
spark.stop()